In [25]:
import numpy as np
import numpy.random as rnd
import numpy.linalg as linalg
import math
from scipy.special import softplus
import sys

Necessary functions for the AMP loop functions

AMP loop functions

In [26]:
def InputChannel_BO(R, Sigma):
    SigmaInv = linalg.inv(Sigma)
    v = linalg.inv(SigmaInv + np.eye(2))
    return v @ SigmaInv @ R, v

def GOutput_BO(omega, Y, V, IntLimits):
    Vinv = linalg.inv(V)
    return Vinv @ np.array([Y - omega[0], V[0,1]*(Y - omega[0])/V[0,0]]), Vinv @ np.array([[-1, 0],[-V[0,1]/V[0,0], 0]])

def OutputChannel_BO(g, Dg, a, Xi, X2i):
    Sigma = linalg.inv(-np.einsum("kjl,k->jl", Dg, X2i))
    R = a + Sigma @ np.einsum("ki,k->i", g, Xi)
    return R, Sigma

AMP runner function

In [27]:
def GAMP_BO(X, Y, MaxIter = 1e4, EpsConvergence = 1e-6, Verbose = False, VerboseRate = 100, IntLimits = 20):
    X2 = X*X
    M = X.shape[0]
    d = X.shape[1]

    # Initializing variables
    a = np.zeros((d, 2))
    v = np.tile(np.eye(2), (d, 1, 1))
    g = np.zeros((M, 2))
    NIter = 0
    Conv = 1

    while((Conv > EpsConvergence) and (NIter < MaxIter)):
        # Updating mean and variance
        V = np.einsum("ki,ijl->kjl", X2, v)
        omega = np.einsum("il,ki->kl", a, X) - np.einsum("kjl,kl->kj", V, g)
        newg, newDg = zip(*[GOutput_BO(omega[k,:], Y[k], V[k,:,:], IntLimits) for k in range(len(omega))])
        # Updating output channel
        g = np.array(newg)
        Dg = np.array(newDg)
        print(g)
        print(Dg)
        newR, newSigma = zip(*[OutputChannel_BO(g, Dg, a[i,:], X[:,i], X2[:,i]) for i in range(len(a))])
        R = np.array(newR)
        Sigma = np.array(newSigma)
        # Updating input channel
        newa, newv = zip(*[InputChannel_BO(R[i], Sigma[i]) for i in range(len(R))])
        Conv = np.sum(np.abs(a - np.array(newa)))
        a = np.array(newa)
        v = np.array(newv)
        if(Verbose and NIter%VerboseRate == 0):
            print("Iteration %s" % NIter)
            print("Current convergence criterion %s" % Conv)
        NIter += 1
    return a, v

Main

In [28]:
d = 100
M = 200
X = rnd.normal(0, 1/np.sqrt(d), size = (M, d))
wvTrue = rnd.normal(0, 1, size = (d, 2))
yTrue = np.matmul(X, wvTrue[:,0])
yNoisy = yTrue
a, v = GAMP_BO(X, yNoisy, MaxIter = 100, EpsConvergence = 1e-6, Verbose = True, VerboseRate = 1, IntLimits = 30)

[[-0.52285338  0.        ]
 [-0.26252801  0.        ]
 [-0.36516575  0.        ]
 [-0.19903884  0.        ]
 [ 1.98073359  0.        ]
 [-1.38052049  0.        ]
 [ 1.03457351  0.        ]
 [-0.17095547  0.        ]
 [-1.11928517  0.        ]
 [ 1.47667155  0.        ]
 [ 0.87327763  0.        ]
 [-1.99175546  0.        ]
 [-1.46763148  0.        ]
 [-0.97183456  0.        ]
 [ 0.66597049  0.        ]
 [-1.72126002  0.        ]
 [-1.56459541  0.        ]
 [-0.28524034  0.        ]
 [-0.67909803  0.        ]
 [-0.29929154  0.        ]
 [-1.16593046  0.        ]
 [ 2.39822639  0.        ]
 [-1.00957794  0.        ]
 [ 1.31457515  0.        ]
 [-0.78716395  0.        ]
 [ 1.28565409  0.        ]
 [-0.37120441  0.        ]
 [ 0.63928003  0.        ]
 [ 1.24226507  0.        ]
 [ 0.29673669  0.        ]
 [ 0.19713923  0.        ]
 [-0.97035599  0.        ]
 [ 0.54742065  0.        ]
 [ 0.80994432  0.        ]
 [ 0.59643775  0.        ]
 [ 0.1862524   0.        ]
 [ 0.05591377  0.        ]
 

LinAlgError: Singular matrix